# 01 - Feature Engineering

**Goal:** Build feature matrices (one per position) that every subsequent modelling phase depends on.

## 1. Database Connection

In [5]:
import sys
import os

# Add project root to path so we can import src modules
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from src.pg_config import get_pg_config

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Build SQLAlchemy engine from existing project config
cfg = get_pg_config()
engine = create_engine(
    f'postgresql+psycopg2://{cfg.user}:{cfg.password}@{cfg.host}:{cfg.port}/{cfg.dbname}'
)
print(f'Connected to {cfg.dbname} on {cfg.host}:{cfg.port}')

Connected to fpl_db on localhost:7651


## 2. Load Raw Data from Gold Schema

In [6]:
# Primary data source: per-player, per-gameweek stats
history = pd.read_sql('SELECT * FROM gold.players_history ORDER BY element, round', engine)

# Player metadata (need element_type for position)
players = pd.read_sql('SELECT * FROM gold.players', engine)

# Team strength ratings
teams = pd.read_sql('SELECT * FROM gold.teams', engine)

# Fixture difficulty
fixtures = pd.read_sql('SELECT * FROM gold.fixtures_main', engine)

print(f'history:  {history.shape[0]:,} rows x {history.shape[1]} cols')
print(f'players:  {players.shape[0]:,} rows x {players.shape[1]} cols')
print(f'teams:    {teams.shape[0]:,} rows x {teams.shape[1]} cols')
print(f'fixtures: {fixtures.shape[0]:,} rows x {fixtures.shape[1]} cols')

history:  23,165 rows x 41 cols
players:  822 rows x 88 cols
teams:    20 rows x 13 cols
fixtures: 380 rows x 12 cols


In [7]:
# Quick look at the history table — this is our primary dataset
history.head(3)

,element,fixture,opponent_team,total_points,was_home,kickoff_time,team_h_score,team_a_score,round,modified,minutes,goals_scored,assists,clean_sheets,goals_conceded,own_goals,penalties_saved,penalties_missed,yellow_cards,red_cards,saves,bonus,bps,influence,creativity,threat,ict_index,clearances_blocks_interceptions,recoveries,tackles,defensive_contribution,starts,expected_goals,expected_assists,expected_goal_involvements,expected_goals_conceded,value,transfers_balance,selected,transfers_in,transfers_out
0,1,9,14,10,False,2025-08-17 15:30:00,0,1,1,False,90,0,0,1,0,0,0,0,1,0,7,3,38,49.2,0.0,0.0,4.9,1,13,0,0,1,0.0,0.00,0.00,1.52,55,0,1531911,0,0
1,1,11,11,6,True,2025-08-23 16:30:00,5,0,2,False,90,0,0,1,0,0,0,0,0,0,1,0,28,13.4,0.0,0.0,1.3,0,3,0,0,1,0.0,0.00,0.00,0.17,55,218659,2284634,277339,58680
2,1,25,12,2,False,2025-08-31 15:30:00,1,0,3,False,90,0,0,0,1,0,0,0,0,0,2,0,12,20.0,10.0,0.0,3.0,0,12,0,0,1,0.0,0.02,0.02,0.52,55,-12311,2406964,146739,159050


In [8]:
history.dtypes

element                                     int64
fixture                                     int64
opponent_team                               int64
total_points                                int64
was_home                                     bool
kickoff_time                       datetime64[ns]
team_h_score                                int64
team_a_score                                int64
round                                       int64
modified                                     bool
minutes                                     int64
goals_scored                                int64
assists                                     int64
clean_sheets                                int64
goals_conceded                              int64
own_goals                                   int64
penalties_saved                             int64
penalties_missed                            int64
yellow_cards                                int64
red_cards                                   int64


In [9]:
# How many unique players and gameweeks?
print(f'Unique players: {history["element"].nunique()}')
print(f'Gameweek range: {history["round"].min()} - {history["round"].max()}')
print(f'Rows with 0 minutes: {(history["minutes"] == 0).sum()} ({(history["minutes"] == 0).mean():.1%})')

Unique players: 822
Gameweek range: 1 - 30
Rows with 0 minutes: 14063 (60.7%)


## 3. Filter: Minutes > 0

Players who didn't play (0 minutes) get 0 points regardless of skill. Predicting *whether* someone plays is a different problem (squad selection, injuries). For now, we only model players who actually played.

In [10]:
df = history[history['minutes'] > 0].copy()
df = df.sort_values(['element', 'round']).reset_index(drop=True)
print(f'After filtering: {df.shape[0]:,} rows ({df.shape[0]/history.shape[0]:.1%} of original)')

After filtering: 9,102 rows (39.3% of original)


## 4. Data Leakage Demonstration

**Data leakage** means the model accidentally sees information from the future during training. This is the #1 mistake in time-series ML.

Without `.shift(1)`, a rolling average for GW 10 would *include* GW 10's result — but at prediction time we don't know GW 10's result yet. The model looks great in training but fails in production.

In [11]:
# Pick one player to demonstrate
demo_player = df[df['element'] == df['element'].iloc[0]].head(8)[['element', 'round', 'total_points']].copy()

# WRONG: rolling without shift — includes current GW
demo_player['rolling_3_WRONG'] = demo_player['total_points'].rolling(3, min_periods=1).mean().round(1)

# RIGHT: shift first, then roll — only uses past GWs
demo_player['rolling_3_RIGHT'] = demo_player['total_points'].shift(1).rolling(3, min_periods=1).mean().round(1)

print('Compare: WRONG includes current GW in the average, RIGHT only uses past GWs')
print('GW1 RIGHT should be NaN (no prior data to use)\n')
demo_player

Compare: WRONG includes current GW in the average, RIGHT only uses past GWs
GW1 RIGHT should be NaN (no prior data to use)



,element,round,total_points,rolling_3_WRONG,rolling_3_RIGHT
0,1,1,10,10.0,NaN
1,1,2,6,8.0,10.0
2,1,3,2,6.0,8.0
3,1,4,6,4.7,6.0
4,1,5,2,3.3,4.7
5,1,6,2,3.3,3.3
6,1,7,6,3.3,3.3
7,1,8,6,4.7,3.3


## 5. Rolling Form Features (3 and 5 GW averages)

These capture *recent* performance. A player's last 3 GW average is more predictive than their season average because form fluctuates.

Method: `groupby('element').shift(1).rolling(N).mean()`
- `groupby('element')` — per player
- `.shift(1)` — exclude current GW (prevents leakage)
- `.rolling(N).mean()` — average of last N values

In [12]:
# Columns to compute rolling averages for
rolling_cols = [
    'total_points', 'minutes', 'goals_scored', 'assists',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'ict_index', 'bps', 'clean_sheets', 'goals_conceded',
    'expected_goals_conceded', 'bonus', 'influence', 'creativity', 'threat'
]

for col in rolling_cols:
    # Cast to float to handle potential integer columns with NaN
    shifted = df.groupby('element')[col].shift(1)
    
    for window in [3, 5]:
        short_name = col.replace('expected_goals', 'xg') \
                       .replace('expected_assists', 'xa') \
                       .replace('expected_goal_involvements', 'xgi') \
                       .replace('expected_goals_conceded', 'xgc') \
                       .replace('total_points', 'pts') \
                       .replace('goals_scored', 'goals') \
                       .replace('clean_sheets', 'cs') \
                       .replace('goals_conceded', 'gc') \
                       .replace('ict_index', 'ict')
        df[f'{short_name}_rolling_{window}'] = shifted.rolling(window, min_periods=1).mean()

print(f'Added {len(rolling_cols) * 2} rolling mean features')
print('Sample rolling columns:', [c for c in df.columns if 'rolling' in c][:6])

Added 32 rolling mean features
Sample rolling columns: ['pts_rolling_3', 'pts_rolling_5', 'minutes_rolling_3', 'minutes_rolling_5', 'goals_rolling_3', 'goals_rolling_5']


## 6. Rolling Standard Deviation

How *consistent* is the player? A player averaging 5 pts with std=1 is more reliable than one averaging 5 with std=4.

In [13]:
std_cols = ['total_points', 'minutes', 'bps']

for col in std_cols:
    shifted = df.groupby('element')[col].shift(1)
    for window in [3, 5]:
        short_name = col.replace('total_points', 'pts').replace('goals_scored', 'goals')
        df[f'{short_name}_std_{window}'] = shifted.rolling(window, min_periods=2).std()

print(f'Added {len(std_cols) * 2} rolling std features')
[c for c in df.columns if '_std_' in c]

Added 6 rolling std features


['pts_std_3',
 'pts_std_5',
 'minutes_std_3',
 'minutes_std_5',
 'bps_std_3',
 'bps_std_5']

## 7. Season Aggregates

These capture the *baseline quality* of a player, separate from recent form.
- `season_avg_pts` — expanding mean (all prior GWs, excluding current)
- `season_total_minutes` — cumulative minutes played
- `games_played` — number of appearances so far

In [14]:
# Expanding mean of points (shift first to exclude current GW)
df['season_avg_pts'] = df.groupby('element')['total_points'].apply(
    lambda x: x.shift(1).expanding().mean()
).reset_index(level=0, drop=True)

# Cumulative minutes (shift so it reflects minutes BEFORE this GW)
df['season_total_minutes'] = df.groupby('element')['minutes'].apply(
    lambda x: x.shift(1).expanding().sum()
).reset_index(level=0, drop=True)

# Games played before this GW
df['games_played'] = df.groupby('element').cumcount()

print('Season aggregates added')
df[['element', 'round', 'total_points', 'minutes', 'season_avg_pts', 'season_total_minutes', 'games_played']].head(10)

Season aggregates added


,element,round,total_points,minutes,season_avg_pts,season_total_minutes,games_played
0,1,1,10,90,NaN,NaN,0
1,1,2,6,90,10.000000,90.0,1
2,1,3,2,90,8.000000,180.0,2
3,1,4,6,90,6.000000,270.0,3
4,1,5,2,90,6.000000,360.0,4
5,1,6,2,90,5.200000,450.0,5
6,1,7,6,90,4.666667,540.0,6
7,1,8,6,90,4.857143,630.0,7
8,1,9,6,90,5.000000,720.0,8
9,1,10,6,90,5.111111,810.0,9


## 8. Opponent Strength

Before each match, we know who the opponent is. Teams have different home/away strength ratings.

For a player:
- If they're playing **at home**, their opponent is the *away* team → use opponent's **away** attack/defence strength
- If they're playing **away**, their opponent is the *home* team → use opponent's **home** attack/defence strength

In [15]:
# Build opponent strength lookup
# When opponent is playing away (player is home): use opponent's away strengths
# When opponent is playing home (player is away): use opponent's home strengths

teams_cols = teams[['id', 'strength', 'strength_overall_home', 'strength_overall_away',
                     'strength_attack_home', 'strength_attack_away',
                     'strength_defence_home', 'strength_defence_away']].copy()

# For rows where player is HOME → opponent is AWAY
home_merge = teams_cols.rename(columns={
    'id': 'opponent_team',
    'strength': 'opp_strength',
    'strength_overall_away': 'opp_strength_overall',
    'strength_attack_away': 'opp_strength_attack',
    'strength_defence_away': 'opp_strength_defence'
})[['opponent_team', 'opp_strength', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']]

# For rows where player is AWAY → opponent is HOME
away_merge = teams_cols.rename(columns={
    'id': 'opponent_team',
    'strength': 'opp_strength',
    'strength_overall_home': 'opp_strength_overall',
    'strength_attack_home': 'opp_strength_attack',
    'strength_defence_home': 'opp_strength_defence'
})[['opponent_team', 'opp_strength', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']]

# Split, merge, recombine
df_home = df[df['was_home'] == True].merge(home_merge, on='opponent_team', how='left')
df_away = df[df['was_home'] == False].merge(away_merge, on='opponent_team', how='left')
df = pd.concat([df_home, df_away]).sort_values(['element', 'round']).reset_index(drop=True)

print('Opponent strength features added')
df[['element', 'round', 'opponent_team', 'was_home', 'opp_strength_overall', 'opp_strength_attack', 'opp_strength_defence']].head(6)

Opponent strength features added


,element,round,opponent_team,was_home,opp_strength_overall,opp_strength_attack,opp_strength_defence
0,1,1,14,False,1180,1100,1260
1,1,2,11,True,1200,1140,1260
2,1,3,12,False,1215,1170,1260
3,1,4,16,True,1145,1120,1170
4,1,5,13,True,1350,1300,1400
5,1,6,15,False,1150,1150,1150


## 9. Fixture Context

- `is_home` — binary flag (home advantage is real in football)
- `fixture_difficulty` — FDR rating (1-5) from fixtures table. 1 = easy, 5 = hardest.

In [16]:
# is_home: convert boolean to int for the model
df['is_home'] = df['was_home'].astype(int)

# Fixture difficulty: need to figure out which team the player belongs to
# fixtures_main has team_h, team_a, team_h_difficulty, team_a_difficulty
# For home players: use team_h_difficulty; for away players: use team_a_difficulty

# We need to join on fixture id
fixture_diff = fixtures[['id', 'team_h_difficulty', 'team_a_difficulty']].rename(
    columns={'id': 'fixture'}
)

df = df.merge(fixture_diff, on='fixture', how='left')

# Select the right difficulty based on whether player was home or away
df['fixture_difficulty'] = np.where(
    df['was_home'],
    df['team_h_difficulty'],
    df['team_a_difficulty']
)

# Drop the intermediate columns
df.drop(columns=['team_h_difficulty', 'team_a_difficulty'], inplace=True)

print('Fixture context added')
df[['element', 'round', 'was_home', 'is_home', 'fixture_difficulty']].head(6)

Fixture context added


,element,round,was_home,is_home,fixture_difficulty
0,1,1,False,0,4
1,1,2,True,1,2
2,1,3,False,0,4
3,1,4,True,1,2
4,1,5,True,1,4
5,1,6,False,0,4


## 10. Player Value Features

- `value` — FPL price this GW (already in history table, in tenths: 45 = 4.5m)
- `value_change_3gw` — price change over last 3 GWs (rising price = crowd thinks player is performing)

In [24]:
# value is already in the dataframe from players_history
# Compute 3GW value change (current value minus value 3 GWs ago)
df['value_change_3gw'] = df.groupby('element')['value'].diff(3)
df['value_change_3gw'] = df['value_change_3gw'].fillna(0)

print('Value features added')
df[['element', 'round', 'value', 'value_change_3gw']].head(8)

Value features added


,element,round,value,value_change_3gw
0,1,1,55,0.0
1,1,2,55,0.0
2,1,3,55,0.0
3,1,4,55,0.0
4,1,5,55,0.0
5,1,6,56,1.0
6,1,7,56,1.0
7,1,8,57,2.0


## 11. Add Position from Players Table

We need `element_type` to split into position-specific models. The players table has this mapped to human-readable names.

In [35]:
players

,ID,First Name,Second Name,chance_of_playing_next_round,chance_of_playing_this_round,code,cost_change_event,cost_change_event_fall,cost_change_start,cost_change_start_fall,dreamteam_count,Position,ep_next,ep_this,event_points,form,in_dreamteam,news,news_added,Price,Points Per Game,selected_by_percent,status,Team,Points,transfers_in,transfers_in_event,transfers_out,transfers_out_event,value_form,...,xG,xA,xGI,xGC,influence_rank,influence_rank_type,threat_rank,threat_rank_type,ict_index_rank_type,corners_and_indirect_freekicks_order,direct_freekicks_order,penalties_order,xG90,Saves90,xA90,xGI90,xGC90,GC90,now_cost_rank,now_cost_rank_type,form_rank,form_rank_type,points_per_game_rank_type,selected_rank,selected_rank_type,starts_per_90,CS90,DC90,Goals90,Assists90
0,1,David,Raya Martín,NaN,NaN,154561,0,0,5,-5,1,Goalkeeper,0.0,5.2,7,4.2,True,,NaT,6.0,4.2,31.5,a,Arsenal,129,4117165,27470,2446655,75509,0.7,...,0.0,0.06,0.06,22.30,100,17,813,94,16,NaN,NaN,NaN,0.00,1.61,0.00,0.00,0.72,0.71,86,1,65,5,4,9,1,1.0,0.48,0.00,0.00,0.00
1,2,Kepa,Arrizabalaga Revuelta,NaN,NaN,109745,-1,1,-5,5,0,Goalkeeper,0.0,1.0,0,0.0,False,,NaT,4.0,0.0,0.4,a,Arsenal,0,14730,79,77353,375,0.0,...,0.0,0.00,0.00,0.00,570,66,529,38,66,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,667,43,454,58,66,283,38,0.0,0.00,0.00,0.00,0.00
2,3,Karl,Hein,0.0,0.0,463748,0,0,0,0,0,Goalkeeper,0.0,0.0,0,0.0,False,Has joined Werder Bremen on loan for the rest ...,2025-08-26 13:44:02.357864,4.0,0.0,0.2,u,Arsenal,0,5545,0,48287,55,0.0,...,0.0,0.00,0.00,0.00,587,75,548,49,75,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,688,54,472,68,75,373,54,0.0,0.00,0.00,0.00,0.00
3,4,Tommy,Setford,NaN,NaN,551221,0,0,-1,1,0,Goalkeeper,0.0,1.0,0,0.0,False,,NaT,3.9,0.0,0.2,a,Arsenal,0,33905,62,34060,331,0.0,...,0.0,0.00,0.00,0.00,559,60,515,30,60,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,777,89,442,51,60,363,52,0.0,0.00,0.00,0.00,0.00
4,5,Gabriel,dos Santos Magalhães,100.0,100.0,226597,0,0,12,-12,5,Defender,0.0,8.0,9,7.0,True,,2025-11-17 12:00:07.436586,7.2,6.9,37.8,a,Arsenal,173,8257937,37268,5346948,138670,1.0,...,1.9,1.65,3.55,15.82,19,7,140,26,24,NaN,NaN,NaN,0.08,0.00,0.07,0.15,0.66,0.62,32,1,7,3,1,5,1,1.0,0.58,9.31,0.12,0.17
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
817,818,James,Wilson,NaN,NaN,613232,0,0,0,0,0,Forward,0.0,-0.5,0,0.0,False,,NaT,4.5,0.0,0.0,a,Spurs,0,2638,401,894,201,0.0,...,0.0,0.00,0.00,0.00,747,81,728,80,81,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,500,71,677,70,81,673,86,0.0,0.00,0.00,0.00,0.00
818,819,Ollie,Shield,NaN,NaN,494556,0,0,0,0,0,Defender,0.0,0.5,0,0.0,False,,NaT,4.0,0.0,0.0,a,Brentford,0,252,45,59,20,0.0,...,0.0,0.00,0.00,0.00,594,222,555,218,222,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,696,177,479,187,223,774,260,0.0,0.00,0.00,0.00,0.00
819,820,Kian,McMahon-Brown,NaN,NaN,644593,0,0,0,0,0,Midfielder,-0.5,-0.5,0,0.0,False,,NaT,4.5,0.0,0.0,a,Burnley,0,93,22,17,11,0.0,...,0.0,0.00,0.00,0.00,753,319,734,315,319,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,508,315,684,274,320,814,362,0.0,0.00,0.00,0.00,0.00
820,821,Jack,Whittaker,NaN,NaN,570929,0,0,0,0,0,Midfielder,1.3,1.3,0,0.0,False,,NaT,4.5,0.0,0.0,a,Sunderland,0,15,15,0,0,0.0,...,0.0,0.00,0.00,0.00,725,298,706,294,299,NaN,NaN,NaN,0.00,0.00,0.00,0.00,0.00,0.00,472,289,653,251,301,821,369,0.0,0.00,0.00,0.00,0.00


In [37]:
# The players table uses aliased column names — "ID" and "Position"
position_map = players[['ID', 'Position']].rename(columns={'ID': 'element'})
df = df.merge(position_map, on='element', how='left')
print('Position distribution:')
#df['Position'].value_counts()

MergeError: Passing 'suffixes' which cause duplicate columns {'Position_x'} is not allowed.

## 13. Define Feature Set and Target

Now we select which columns are features (inputs to the model) and which is the target (what we predict).

In [26]:
TARGET = 'total_points'

# Identifiers (not features, but needed for debugging/splitting)
ID_COLS = ['element', 'round', 'Position', 'kickoff_time']

# Raw columns from history that should NOT be features
# (they are either the target, identifiers, or would cause leakage)
EXCLUDE_COLS = [
    'total_points',       # target
    'fixture',            # identifier
    'opponent_team',      # used to build opp_strength, not a feature itself
    'was_home',           # replaced by is_home
    'kickoff_time',       # used to build days_since_last_match
    'team_h_score',       # match result — leakage!
    'team_a_score',       # match result — leakage!
    'modified',           # metadata
    'element',            # identifier
    'round',              # will use for temporal split, not as feature
    'Position',           # used for splitting, not as feature within position models
    'days_since_last_match', #does not include cups etc..
    # Raw per-GW stats are also leakage if used as features
    # (they describe THIS gameweek's outcome, not prior info)
    'minutes', 'goals_scored', 'assists', 'clean_sheets', 'goals_conceded',
    'own_goals', 'penalties_saved', 'penalties_missed', 'yellow_cards',
    'red_cards', 'saves', 'bonus', 'bps', 'influence', 'creativity',
    'threat', 'ict_index', 'clearances_blocks_interceptions', 'recoveries',
    'tackles', 'defensive_contribution', 'starts',
    'expected_goals', 'expected_assists', 'expected_goal_involvements',
    'expected_goals_conceded',
    'transfers_balance', 'selected', 'transfers_in', 'transfers_out',
]

FEATURE_COLS = [c for c in df.columns if c not in EXCLUDE_COLS]

print(f'Target: {TARGET}')
print(f'Features ({len(FEATURE_COLS)}):')
for i, col in enumerate(sorted(FEATURE_COLS)):
    print(f'  {i+1:2d}. {col}')

Target: total_points
Features (51):
   1. Position_x
   2. Position_y
   3. assists_rolling_3
   4. assists_rolling_5
   5. bonus_rolling_3
   6. bonus_rolling_5
   7. bps_rolling_3
   8. bps_rolling_5
   9. bps_std_3
  10. bps_std_5
  11. creativity_rolling_3
  12. creativity_rolling_5
  13. cs_rolling_3
  14. cs_rolling_5
  15. fixture_difficulty
  16. games_played
  17. gc_rolling_3
  18. gc_rolling_5
  19. goals_rolling_3
  20. goals_rolling_5
  21. ict_rolling_3
  22. ict_rolling_5
  23. influence_rolling_3
  24. influence_rolling_5
  25. is_home
  26. minutes_rolling_3
  27. minutes_rolling_5
  28. minutes_std_3
  29. minutes_std_5
  30. opp_strength
  31. opp_strength_attack
  32. opp_strength_defence
  33. opp_strength_overall
  34. pts_rolling_3
  35. pts_rolling_5
  36. pts_std_3
  37. pts_std_5
  38. season_avg_pts
  39. season_total_minutes
  40. threat_rolling_3
  41. threat_rolling_5
  42. value
  43. value_change_3gw
  44. xa_rolling_3
  45. xa_rolling_5
  46. xg_concede

## 14. Inspect Nulls

Rolling features will be NaN for early gameweeks (not enough history). This is expected — those rows will be dropped.

In [27]:
null_counts = df[FEATURE_COLS + [TARGET]].isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)

null_summary = pd.DataFrame({'nulls': null_counts, 'pct': null_pct})
null_summary = null_summary[null_summary['nulls'] > 0].sort_values('pct', ascending=False)

print(f'Features with nulls: {len(null_summary)} / {len(FEATURE_COLS)}')
print(f'Total rows: {len(df):,}\n')
null_summary

Features with nulls: 40 / 51
Total rows: 9,102



,nulls,pct
season_total_minutes,521,5.7
season_avg_pts,521,5.7
minutes_std_3,77,0.8
pts_std_3,77,0.8
bps_std_3,77,0.8
pts_std_5,5,0.1
minutes_std_5,5,0.1
bps_std_5,5,0.1
assists_rolling_5,1,0.0
assists_rolling_3,4,0.0


In [32]:
# Drop rows where ANY rolling feature is NaN
# This removes early-season rows where we don't have enough history
rows_before = len(df)
df_clean = df.dropna(subset=FEATURE_COLS).copy()
rows_after = len(df_clean)

print(f'Dropped {rows_before - rows_after:,} rows with NaN features ({(rows_before - rows_after)/rows_before:.1%})')
print(f'Remaining: {rows_after:,} rows')

Dropped 550 rows with NaN features (6.0%)
Remaining: 8,552 rows


## 15. Spot-Check: Verify Rolling Averages

Pick a known player and manually verify the rolling calculations are correct.

In [39]:
# Pick the player with the most rows for a good demonstration
top_player = df_clean.groupby('element').size().idxmax()
player_name = players.loc[players['ID'] == top_player, 'Second Name'].iloc[0]

check = df_clean[df_clean['element'] == top_player][[
    'element', 'round', 'total_points', 'pts_rolling_3', 'pts_rolling_5',
    'season_avg_pts', 'games_played'
]].head(10)

print(f'Spot-check: {player_name} (element={top_player})')
print('Verify: pts_rolling_3 for row N should be mean of total_points from rows N-3 to N-1')
print()
check

Spot-check: Zubimendi Ibáñez (element=26)
Verify: pts_rolling_3 for row N should be mean of total_points from rows N-3 to N-1



,element,round,total_points,pts_rolling_3,pts_rolling_5,season_avg_pts,games_played
341,26,2,3,3.000000,2.00,5.000000,1
342,26,3,1,4.000000,2.50,4.000000,2
343,26,4,16,3.000000,2.50,3.000000,3
344,26,5,2,6.666667,6.25,6.250000,4
345,26,6,2,6.333333,5.40,5.400000,5
346,26,7,3,6.666667,4.80,4.833333,6
347,26,8,3,2.333333,4.80,4.571429,7
348,26,9,5,2.666667,5.20,4.375000,8
349,26,10,3,3.666667,3.00,4.444444,9
350,26,11,5,3.666667,3.20,4.300000,10


## 16. Split by Position and Save to Parquet

Positions score fundamentally differently:
- **GK**: clean sheets, saves, penalty saves
- **DEF**: clean sheets, goals (from set pieces)
- **MID**: goals, assists, clean sheets (for some)
- **FWD**: goals, assists only

Separate models let each position learn its own patterns.

In [41]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'data')

positions = {'Goalkeeper': 'gk', 'Defender': 'def', 'Midfielder': 'mid', 'Forward': 'fwd'}

for pos_name, pos_short in positions.items():
    pos_df = df_clean[df_clean['Position'] == pos_name].copy()
    
    filepath = os.path.join(DATA_DIR, f'features_{pos_short}.parquet')
    pos_df.to_parquet(filepath, index=False)
    
    print(f'{pos_name:12s} | {pos_df.shape[0]:5,} rows | {pos_df.shape[1]:3d} cols | saved to features_{pos_short}.parquet')

# Also save the full dataset and the feature column list
df_clean.to_parquet(os.path.join(DATA_DIR, 'features_all.parquet'), index=False)

import json
with open(os.path.join(DATA_DIR, 'feature_columns.json'), 'w') as f:
    json.dump(FEATURE_COLS, f, indent=2)

print(f'\nAll positions  | {df_clean.shape[0]:5,} rows | saved to features_all.parquet')
print(f'Feature list   | {len(FEATURE_COLS)} features | saved to feature_columns.json')

KeyError: 'Position'

## Summary

Feature categories built:
1. **Rolling form** (3 & 5 GW) — pts, minutes, goals, assists, xG, xA, xGI, ICT, BPS, clean sheets, goals conceded, xGC, bonus, influence, creativity, threat
2. **Rolling std** (3 & 5 GW) — pts, minutes, BPS
3. **Season aggregates** — avg pts, total minutes, games played
4. **Opponent strength** — overall, attack, defence (home/away adjusted)
5. **Fixture context** — is_home, fixture difficulty (FDR)
6. **Player value** — current value, 3GW value change
7. **Rest** — days since last match

**Leakage prevention:**
- All rolling features use `.shift(1)` before `.rolling()`
- Raw per-GW stats (goals, assists, etc.) excluded from features
- Match results (team_h_score, team_a_score) excluded
- Target (total_points) never used as input

Next: `02_eda.ipynb` — explore these features before modeling.